# Week 34, Day 6: Forge

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w034d6.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
count = 0

def visit():
    global count
    count += 1
    return count

def report():
    return f"{count} visits"

visit()
visit()
print(report(), visit(), report())

**Predict before you run:** what will this print?



In [ ]:
handlers = {}
for name in ["a", "b", "c"]:
    handlers[name] = lambda: name

print([handlers[k]() for k in "abc"])

**Predict before you run:** what will this print?



In [ ]:
class Bird:
    def move(self):
        return "flies"

class Penguin(Bird):
    def move(self):
        raise NotImplementedError("penguins can't fly")

for b in [Bird(), Penguin()]:
    try:
        print(b.move())
    except NotImplementedError as err:
        print("broken promise:", err)

**Predict before you run:** what will this print?



In [ ]:
LIMIT = 3

def ok(n):
    return n <= LIMIT

def warn(n):
    return n > LIMIT

LIMIT = 5
print(ok(4), warn(4))

**Predict before you run:** what will this print?



In [ ]:
def before(x):
    if x is not None:
        if x > 0:
            return "positive"
    return "other"

def after(x):
    if x is None or x <= 0:
        return "other"
    return "positive"

print([before(x) == after(x) for x in [None, -1, 0, 7]])

### Exercise: A raffle you can test

`draw` picks winners with the real `random.choice`, so no test can know who will win. Rewrite it as a class `Raffle(entries, pick)`, where `pick` is any function that takes a list and returns one item from it (like `random.choice`, which should be the default when `pick` isn't given). `draw(n)` returns `n` different winners in the order drawn (removing each from the entries as it's drawn), raising `ValueError` if there aren't enough entries. Entries given to the raffle must not be changed.

In [ ]:
import random

def draw(entries, n):
    winners = []
    for _ in range(n):
        w = random.choice(entries)
        winners.append(w)
        entries.remove(w)
    return winners

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_raffle():\n    \"testable\"\n    R = ns[\"Raffle\"]\n    names = [\"Ada\", \"Bo\", \"Chi\", \"Dami\"]\n    first = lambda items: items[0]\n    last = lambda items: items[-1]\n    assert R(names, first).draw(2) == [\"Ada\", \"Bo\"] and R(names, last).draw(3) == [\"Dami\", \"Chi\", \"Bo\"]\n    assert names == [\"Ada\", \"Bo\", \"Chi\", \"Dami\"], \"the entries passed in must not change\"\n    r = R(names)\n    got = r.draw(4)\n    assert sorted(got) == sorted(names)\n    try:\n        R([\"Ada\"], first).draw(2)\n        assert False, \"too few entries should raise ValueError\"\n    except ValueError:\n        pass")

### Exercise: Wind in a table

Write `describe_wind(mph)` using a table `BEAUFORT`, a list of `(below_mph, description)` pairs, instead of an `if` chain: below 1 is `"calm"`, below 8 `"light breeze"`, below 19 `"moderate breeze"`, below 32 `"strong breeze"`, below 47 `"gale"`, and 47 or more `"storm"`. Negative speeds raise `ValueError`.

In [ ]:
___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_wind():\n    \"a table\"\n    assert [describe_wind(m) for m in [0, 0.9, 1, 7.9, 8, 18, 19, 31, 46, 47, 90]] == [\n        \"calm\", \"calm\", \"light breeze\", \"light breeze\", \"moderate breeze\", \"moderate breeze\", \"strong breeze\", \"strong breeze\", \"gale\", \"storm\", \"storm\"]\n    assert \"elif\" not in source and isinstance(ns.get(\"BEAUFORT\"), list)\n    try:\n        describe_wind(-1)\n        assert False, \"negative speeds should raise ValueError\"\n    except ValueError:\n        pass")

### Exercise: Forge: redesigning a billing script

Redesign the script to the specification above.

In [ ]:
TOTAL = 0

def process(members_text, visits_text):
    global TOTAL
    members = {}
    for line in members_text.strip().splitlines()[1:]:
        name, plan, student = [p.strip() for p in line.split(",")]
        members[name] = (plan, student == "yes")
    visits = {}
    for line in visits_text.strip().splitlines()[1:]:
        name, date = [p.strip() for p in line.split(",")]
        visits[name] = visits.get(name, 0) + 1
    for name in sorted(members):
        plan, student = members[name]
        n = visits.get(name, 0)
        if plan == "basic":
            fee = 1500 + max(0, n - 4) * 300
        elif plan == "standard":
            fee = 2500 + max(0, n - 12) * 200
        elif plan == "unlimited":
            fee = 3900
        else:
            raise ValueError("unknown plan " + plan)
        if student:
            fee = fee - fee * 20 // 100
        TOTAL += fee
        print(f"To {name}: {n} visits on {plan}, £{fee / 100:.2f}")
    print(f"Total: £{TOTAL / 100:.2f}")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("MEMBERS = \"\"\"name,plan,student\nAda,basic,no\nBo,standard,yes\nChi,unlimited,no\nDami,basic,yes\n\"\"\"\nVISITS = \"\"\"name,date\nAda,2026-11-01\nAda,2026-11-03\nAda,2026-11-05\nAda,2026-11-08\nAda,2026-11-10\nAda,2026-11-12\nBo,2026-11-02\nChi,2026-11-02\nChi,2026-11-04\n\"\"\"\nWANT = [\"To Ada: 6 visits on basic, \u00a321.00\", \"To Bo: 1 visits on standard, \u00a320.00\", \"To Chi: 2 visits on unlimited, \u00a339.00\",\n        \"To Dami: 0 visits on basic, \u00a312.00\", \"Total: \u00a392.00\"]\ndef test_same_output():\n    \"the same output, twice\"\n    lines = []\n    total = main(MEMBERS, VISITS, lines.append)\n    assert lines == WANT, \"got:\\n\" + \"\\n\".join(lines)\n    again = []\n    assert main(MEMBERS, VISITS, again.append) == total == 9200 and again == WANT, \"a second run should give the same result\"\ndef test_parts():\n    \"the parts\"\n    P, M, B = ns[\"Plan\"], ns[\"Member\"], ns[\"Biller\"]\n    assert PLANS[\"basic\"] == P(\"basic\", 1500, 4, 300) and PLANS[\"unlimited\"].fee(99) == 3900 and PLANS[\"standard\"].fee(14) == 2900\n    members = parse_members(MEMBERS)\n    assert members[1] == M(\"Bo\", \"standard\", True) and len(members) == 4\n    assert count_visits(VISITS) == {\"Ada\": 6, \"Bo\": 1, \"Chi\": 2} and count_visits(\"name,date\\n\") == {}\n    assert monthly_fee(M(\"x\", \"basic\", True), 6, PLANS) == 1680 and money(1680) == \"\u00a316.80\" and money(5) == \"\u00a30.05\"\n    sent = []\n    total = B(PLANS, lambda n, m: sent.append((n, m))).run(members, count_visits(VISITS))\n    assert total == 9200 and sent[0] == (\"Ada\", \"6 visits on basic, \u00a321.00\")\n    for bad in [\"name,plan,student\\nAda,basic\\n\", \"name,plan,student\\nAda,basic,maybe\\n\"]:\n        try:\n            parse_members(bad)\n            assert False, \"should raise ValueError\"\n        except ValueError:\n            pass\n    try:\n        monthly_fee(M(\"x\", \"gold\", False), 1, PLANS)\n        assert False, \"an unknown plan should raise ValueError\"\n    except ValueError:\n        pass\ndef test_design():\n    \"the design\"\n    P, M = ns[\"Plan\"], ns[\"Member\"]\n    assert \"global\" not in source and \"print(\" not in source and \"TOTAL\" not in source and \"elif\" not in source\n    PLANS[\"weekend\"] = P(\"weekend\", 1000, 2, 400)\n    try:\n        assert monthly_fee(M(\"x\", \"weekend\", False), 5, PLANS) == 2200, \"a new plan should only need a new entry\"\n    finally:\n        del PLANS[\"weekend\"]\n    assert source.count(\"/ 100\") <= 2, \"format money in one place (and work out the discount in one place)\"")

> **What changed, and what didn't.** The output didn't change at all: that's what made this a redesign you could trust. What changed is how much each future change costs. A new plan is one line in `PLANS`. A new message format touches only `Biller.run`; emailing instead of printing only touches what's passed as `write`; a new discount only touches `monthly_fee`. The global total is gone, so a second run can't be wrong. And every part can be tested with plain values in a fraction of a second. Notice too what you **didn't** add: no plugin system, no abstract base classes, no configuration files. The plans really vary, so they got an extension point; nothing else did.

### Exercise: Forge bonus: a discount engine

A shop runs promotions that come and go. Build a checkout where each promotion is a pluggable rule and the time is passed in, so time-limited promotions can be tested.

- A basket is a dictionary from item name to `(qty, pence)`. `subtotal(basket)` is the sum of `qty * pence`.
- `Rule`, a Protocol: an attribute `name: str`, and methods `applies(basket, now) -> bool` and `amount(basket, now) -> int` (pence off).
- `PercentOff(name, percent, min_spend=0)`: applies when the subtotal is at least `min_spend`; takes `percent` per cent off the subtotal, rounded down.
- `MultiBuy(name, item, buy, free)`: "buy 3, get 1 free" style. Applies when the basket has at least `buy + free` of the item; for each complete group of `buy + free`, `free` of them cost nothing.
- `HappyHour(name, start, end, percent)`: applies when `now`'s hour is from `start` up to (not including) `end`; takes `percent` off the subtotal.
- `Checkout(rules, clock)`: `clock` is a function returning the current `datetime`. `total(basket)` returns `(pence, names)`: the subtotal minus every applicable rule's amount (each worked out from the original basket), never below 0, and the names of the rules that applied, in the order given. Read the clock **once** per `total`.

Nothing in `Checkout` may name a particular rule class.

In [ ]:
from dataclasses import dataclass
from datetime import datetime
from typing import Protocol

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("BASKET = {\"rice\": (2, 450), \"zobo\": (4, 150), \"suya\": (1, 1500)}\ndef at(h):\n    from datetime import datetime as dt\n    return lambda: dt(2026, 11, 6, h, 0)\ndef test_rules():\n    \"the rules\"\n    P, M, H = ns[\"PercentOff\"], ns[\"MultiBuy\"], ns[\"HappyHour\"]\n    now = at(12)()\n    assert subtotal(BASKET) == 3000 and subtotal({}) == 0\n    assert P(\"ten\", 10).applies(BASKET, now) and P(\"ten\", 10).amount(BASKET, now) == 300 and not P(\"big\", 10, 5000).applies(BASKET, now)\n    m = M(\"zobo deal\", \"zobo\", 3, 1)\n    assert m.applies(BASKET, now) and m.amount(BASKET, now) == 150 and m.amount({\"zobo\": (9, 150)}, now) == 300 and not m.applies({\"zobo\": (3, 150)}, now)\n    h = H(\"lunch\", 12, 14, 20)\n    assert h.applies(BASKET, at(12)()) and h.applies(BASKET, at(13)()) and not h.applies(BASKET, at(14)()) and h.amount(BASKET, now) == 600\ndef test_checkout():\n    \"the checkout\"\n    P, M, H, C = ns[\"PercentOff\"], ns[\"MultiBuy\"], ns[\"HappyHour\"], ns[\"Checkout\"]\n    rules = [P(\"ten off over \u00a325\", 10, 2500), M(\"zobo 3+1\", \"zobo\", 3, 1), H(\"lunch\", 12, 14, 20)]\n    assert C(rules, at(12)).total(BASKET) == (1950, [\"ten off over \u00a325\", \"zobo 3+1\", \"lunch\"])\n    assert C(rules, at(18)).total(BASKET) == (2550, [\"ten off over \u00a325\", \"zobo 3+1\"])\n    assert C([P(\"all\", 100), P(\"again\", 50)], at(9)).total(BASKET) == (0, [\"all\", \"again\"]) and C([], at(9)).total({}) == (0, [])\n    calls = []\n    def clock():\n        calls.append(1)\n        return at(12)()\n    C(rules, clock).total(BASKET)\n    assert len(calls) == 1, \"read the clock once per total\"\n    class Free:\n        name = \"staff\"\n        def applies(self, basket, now):\n            return True\n        def amount(self, basket, now):\n            return 100\n    assert C([Free()], at(9)).total(BASKET) == (2900, [\"staff\"])\n    import ast\n    tree = ast.parse(source)\n    co = next(n for n in ast.walk(tree) if isinstance(n, ast.ClassDef) and n.name == \"Checkout\")\n    seg = ast.get_source_segment(source, co)\n    assert not any(n in seg for n in [\"PercentOff\", \"MultiBuy\", \"HappyHour\"]), \"Checkout shouldn't name rule classes\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Warm-up

If any surprised you, revisit [cohesion and coupling](w034d1.html), [S and O](w034d2.html), [L, I and D](w034d3.html), [DRY, KISS and YAGNI](w034d4.html) or [refactoring](w034d5.html).

## Mixed review

## Forge: redesigning a billing script

A small gym bills its members monthly with the script in the starter. It works, mostly, but it has every problem from this week:

- one function parses two files, works out every fee, formats messages and prints them (low cohesion);
- the fee rules are an `if`/`elif` chain on the plan's name, so a new plan means editing the billing code (open-closed);
- the money format and the numbers for each plan are buried in the logic (DRY);
- it adds to a **global** total, so running it twice in one program reports the wrong total the second time (coupling);
- it prints directly, so it can't be tested without capturing the screen, and can't send messages any other way (dependency inversion).

Redesign it into these parts, with **exactly the same output** as the first run of the old script:

- `Plan`, a frozen dataclass: `name`, `monthly` (pence), `included` (visits included each month) and `per_visit` (pence for each visit beyond those). A method `fee(visits)` gives `monthly + per_visit * max(0, visits - included)`.
- `PLANS`, a dictionary from plan name to `Plan`: `basic` (1500, 4 included, 300 per extra visit), `standard` (2500, 12, 200) and `unlimited` (3900, 0, 0). A new plan is just a new entry.
- `Member`, a frozen dataclass: `name`, `plan` (the plan's name) and `student` (a `bool`).
- `parse_members(text)`: the members file (header `name,plan,student`, where `student` is `yes` or `no`) as a list of `Member`; a line with the wrong number of fields, or `student` not `yes` or `no`, raises `ValueError`. Pure.
- `count_visits(text)`: the visits file (header `name,date`) as a dictionary from name to number of visits. Pure.
- `money(pence)`: like `£12.50`. The only place money is formatted.
- `monthly_fee(member, visits, plans)`: the plan's fee, less 20 per cent (rounded down) for students; an unknown plan raises `ValueError`. Pure.
- `Biller(plans, send)`: `send` is any function taking `(name, message)`. `run(members, visits)` sends each member, in alphabetical order, a message like `5 visits on basic, £18.00`, and returns the total of all fees.
- `main(members_text, visits_text, write)`: the thin shell. `write` is any function taking one line of text; `main` writes a line `To Ada: ...` for each member, then `Total: £...`, and returns the total. Calling it twice gives the same result twice.

No function may use `global` or `print`.

## Forge bonus: a discount engine

## Weekly quiz

## Week 34 review

Tick off each one honestly. If one feels shaky, go back and redo two exercises from that day.

- I can explain cohesion and coupling, separate calculation from input and output, and remove shared globals. ([Day 1](w034d1.html))
- I can split a class by responsibility, and add behaviour through polymorphism, a dictionary of functions or objects passed in. ([Day 2](w034d2.html))
- I can spot broken substitution, keep interfaces small, and pass dependencies in so code can be tested with fakes. ([Day 3](w034d3.html))
- I can remove duplicated knowledge, simplify code, and recognise speculative features. ([Day 4](w034d4.html))
- I can write characterisation tests and refactor in small, named, tested steps. ([Day 5](w034d5.html))
- I redesigned a billing script and built a discount engine using all of it. (Today)

Tomorrow is a **rest day**. Next week: **design patterns**, named solutions to problems that keep coming up, starting with Strategy, which you've already been using.